# Task 1 — Data Acquisition

**COMP5339 Data Engineering · Assignment 1 · EV Charger Data Integration & Augmentation**

---

This notebook programmatically retrieves the two source datasets the assignment is built on,
validates them, records their provenance, and profiles their quality.

| # | Dataset | Source | Retrieved via |
|---|---|---|---|
| 1 | EV charging locations in NSW (Dec 2025) | Transport for NSW | data.gov.au CKAN API |
| 2 | ASGS Edition 4 SA4 digital boundaries | Australian Bureau of Statistics | ABS download page (HTML parsing) |

### How to run

Run every cell from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The first
cell installs the required packages. A complete run takes about 15 seconds and downloads
roughly 30 MB into a `data/` folder created next to this notebook.

The notebook is **safe to re-run**: files already downloaded are verified by checksum and
skipped rather than fetched again.

### Contents

1. [Setup and configuration](#setup)
2. [Shared helpers](#helpers) — HTTP, checksums, provenance manifest
3. [Source 1 — Transport for NSW EV chargers](#source1)
4. [Source 2 — ABS SA4 boundaries](#source2)
5. [Provenance manifest](#manifest)
6. [Data quality profile](#profile)
7. [SA4 boundaries in DuckDB](#duckdb)
8. [Summary and outputs](#summary)

### Scope

This notebook covers **Task 1 only**. It is self-contained: it has no local imports and
depends on nothing outside the packages installed in its first cell, so it can be executed on
its own to reproduce the acquisition stage in full.

The datasets it produces under `data/raw/` are the inputs to the cleaning and spatial
integration in Task 2, and the profiling in section 6 establishes the data quality issues that
Task 2 addresses. Each subsequent task is documented in its own notebook, so that every stage
of the pipeline corresponds to one file.

<a id="setup"></a>
## 1. Setup and configuration

### 1.1 Dependencies

Installing from within the notebook makes it reproducible on a fresh machine — including a
hosted environment such as Google Colab — without any prior setup. `pip` skips anything
already present, so re-running this cell is inexpensive.

In [1]:
%pip install -q "requests>=2.32" "beautifulsoup4>=4.13" "lxml>=5.3" "pandas>=2.2" "duckdb>=1.5"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import hashlib
import json
import re
import tempfile
import zipfile
from datetime import date, datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_colwidth", 60)
print("Imports OK")

Imports OK


### 1.2 Configuration

Every path, URL and constant the notebook uses is defined here rather than scattered through
the cells below. If the ABS publishes a new ASGS edition or TfNSW a newer release, this is the
only cell that needs editing.

In [3]:
# --- Paths -----------------------------------------------------------------
# Everything is created relative to the notebook's own folder, so the project is
# portable: unzip it anywhere and it still works.
PROJECT_ROOT = Path.cwd()

DATA_DIR      = PROJECT_ROOT / "data"
RAW_DIR       = DATA_DIR / "raw"           # source files, exactly as downloaded
INTERIM_DIR   = DATA_DIR / "interim"       # intermediate outputs (Tasks 2-3)
PROCESSED_DIR = DATA_DIR / "processed"     # final outputs (Task 4)

RAW_TFNSW_DIR   = RAW_DIR / "tfnsw"
RAW_ABS_DIR     = RAW_DIR / "abs"
SA4_EXTRACT_DIR = RAW_ABS_DIR / "sa4_shapefile"

MANIFEST_PATH = RAW_DIR / "manifest.json"  # provenance record (see section 5)

for directory in (RAW_TFNSW_DIR, RAW_ABS_DIR, INTERIM_DIR, PROCESSED_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# --- Source 1: Transport for NSW EV charging locations ----------------------
# The TfNSW portal requires a free login; data.gov.au mirrors the identical
# resource behind a public CKAN API, so the URL is discovered rather than fixed.
CKAN_API_BASE   = "https://data.gov.au/data/api/3/action"
CKAN_PACKAGE_ID = "nsw-2-ev-charging-locations"

# Assignment brief: "Retrieve the most-recent version from December 2025".
TARGET_RELEASE_YEAR  = 2025
TARGET_RELEASE_MONTH = 12

# TfNSW names each release `ev_YYYYMMDD.csv`; the embedded date is the only
# reliable version marker, because CKAN's `last_modified` field is null here.
TFNSW_CSV_DATE_PATTERN = re.compile(r"ev_(?P<date>\d{8})\.csv$", re.IGNORECASE)

# Used only if data.gov.au is unreachable, so the run stays reproducible.
TFNSW_FALLBACK_URL = (
    "https://opendata.transport.nsw.gov.au/data/dataset/"
    "be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/"
    "7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv"
)

# Checked against the downloaded header so a changed schema is caught here
# rather than deep inside the Task 2 cleaning code.
EXPECTED_TFNSW_COLUMNS = [
    "OBJECTID", "Station_name", "Station_address", "Operator",
    "Number_of_plugs", "Charger_Type", "Charger_rating",
    "Latitude", "Longitude", "LGANAME", "PCODE", "Source",
]

# --- Source 2: ABS ASGS Edition 4 SA4 boundaries ----------------------------
# The ABS offers no API for boundary files, so the download page is parsed.
ABS_BOUNDARY_PAGE_URL = (
    "https://www.abs.gov.au/statistics/standards/"
    "australian-statistical-geography-standard-asgs/"
    "edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files"
)

# Matching a pattern, not a literal filename, means a change of reference year
# or datum (e.g. SA4_2031_AUST_SHP_GDA2020.zip) is handled automatically.
ABS_SA4_ZIP_PATTERN = re.compile(
    r"SA4_(?P<year>\d{4})_AUST_SHP_(?P<datum>GDA\d{4})\.zip$", re.IGNORECASE
)

# A shapefile is a SET of sibling files; these four are mandatory.
REQUIRED_SHAPEFILE_SUFFIXES = {".shp", ".shx", ".dbf", ".prj"}

# --- HTTP behaviour ---------------------------------------------------------
# A descriptive User-Agent is good scraping etiquette: it identifies the client
# to the server operator instead of impersonating a browser.
USER_AGENT = (
    "COMP5339-Assignment1/1.0 (University of Sydney student project; "
    "data engineering coursework)"
)
REQUEST_TIMEOUT      = 60        # seconds to wait for a response
MAX_RETRIES          = 4         # retries for transient failures
RETRY_BACKOFF_FACTOR = 1.5       # exponential backoff: 1.5s, 3s, 6s, 12s
CHUNK_SIZE           = 1 << 16   # 64 KiB - stream large files, don't buffer them

# Paths are printed relative to the project root, so outputs do not depend on
# (or reveal) where the project happens to be unzipped.
print("Project root: . (the folder containing this notebook)")
print(f"Data folder:  {DATA_DIR.relative_to(PROJECT_ROOT)}/")

Project root: . (the folder containing this notebook)
Data folder:  data/


<a id="helpers"></a>
## 2. Shared helpers

Downloading is the one step guaranteed to fail occasionally — networks drop, servers return
503, large files time out. Concentrating that fragility into a few well-behaved functions lets
the acquisition cells below read as though the network were reliable.

In [4]:
def log(message: str) -> None:
    """Print a timestamped progress message."""
    print(f"[{datetime.now():%H:%M:%S}] {message}")


def build_session() -> requests.Session:
    """
    Return a `requests.Session` that automatically retries transient failures.

    A Session (rather than bare `requests.get` calls) reuses the TCP connection
    and applies one retry policy to every request in the run.

    `Retry` deliberately covers only *transient* problems: connection errors and
    the 429/5xx status codes. A 404 is a genuine answer from the server, so it is
    raised immediately instead of being retried four times.
    """
    retry_policy = Retry(
        total=MAX_RETRIES,
        backoff_factor=RETRY_BACKOFF_FACTOR,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET", "HEAD"]),
        raise_on_status=False,
    )
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    adapter = HTTPAdapter(max_retries=retry_policy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def sha256_of_file(path: Path) -> str:
    """
    Compute a file's SHA-256 checksum, reading it in chunks rather than at once.

    The checksum is this project's evidence of reproducibility: two runs that
    produce the same digest downloaded byte-identical data. It is also how the
    notebook decides whether an already-downloaded file can be trusted.
    """
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(CHUNK_SIZE), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_file(session: requests.Session, url: str, destination: Path) -> Path:
    """
    Stream `url` into `destination` and return the path written.

    Two deliberate choices:

    1. `stream=True` writes the response to disk in 64 KiB chunks instead of
       holding it in memory. The ABS archive is ~30 MB, and the same code should
       still work if a future release is far larger.
    2. The download goes to a temporary `.part` file in the same folder and is
       renamed into place only once it completes. `Path.replace` is atomic on a
       single filesystem, so an interrupted run can never leave a half-written
       file that a later cell would happily read as if it were complete.
    """
    destination.parent.mkdir(parents=True, exist_ok=True)
    log(f"Downloading {url.rsplit('/', 1)[-1]}")

    with session.get(url, stream=True, timeout=REQUEST_TIMEOUT) as response:
        response.raise_for_status()

        handle = tempfile.NamedTemporaryFile(
            delete=False, dir=destination.parent, suffix=".part"
        )
        temp_path = Path(handle.name)
        try:
            with handle:
                for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:                      # skip keep-alive chunks
                        handle.write(chunk)
        except BaseException:
            temp_path.unlink(missing_ok=True)      # never leave a stray .part file
            raise

    temp_path.replace(destination)
    log(f"  saved {destination.name} ({destination.stat().st_size:,} bytes)")
    return destination

### 2.1 The provenance manifest

Every file retrieved is recorded in `data/raw/manifest.json` alongside the URL it came from,
*how* that URL was discovered, when it was fetched, its size and its SHA-256 checksum.

The manifest does two jobs:

* **Reproducibility** — a machine-readable record of exactly which version of each source a
  given run used, which is what lets someone else confirm they obtained the same bytes.
* **Caching** — re-running this notebook should not re-download a 30 MB shapefile that is
  already present and unchanged. The recorded checksum lets the notebook *verify* a cached
  file rather than merely assume it is intact.

In [5]:
def utc_now_iso() -> str:
    """Current UTC time as a timezone-aware ISO-8601 string."""
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def load_manifest() -> dict:
    """Read an existing manifest, tolerating a missing or corrupted file."""
    if not MANIFEST_PATH.exists():
        return {}
    try:
        return json.loads(MANIFEST_PATH.read_text(encoding="utf-8")).get("artefacts", {})
    except (json.JSONDecodeError, OSError) as error:
        # A damaged manifest must not stop the run; the cost is re-downloading.
        log(f"WARNING: could not read manifest ({error}); starting a new one.")
        return {}


def record_artefact(manifest, name, *, path, url, source, discovery_method, extra=None):
    """Add (or replace) the manifest entry for one downloaded file."""
    entry = {
        "source": source,
        "discovery_method": discovery_method,
        "url": url,
        # Stored relative to the project root so the manifest is portable.
        "path": str(path.relative_to(PROJECT_ROOT)),
        "bytes": path.stat().st_size,
        "sha256": sha256_of_file(path),
        "retrieved_at": utc_now_iso(),
    }
    entry.update(extra or {})
    manifest[name] = entry
    return entry


def is_unchanged(manifest, name, path: Path) -> bool:
    """
    True if `path` exists and its checksum matches what was recorded for `name`,
    i.e. the local copy can be trusted and the download skipped.
    """
    entry = manifest.get(name)
    if entry is None or not path.exists() or "sha256" not in entry:
        return False
    if sha256_of_file(path) != entry["sha256"]:
        log(f"WARNING: {path.name} does not match its recorded checksum; re-downloading.")
        return False
    return True


def save_manifest(manifest) -> Path:
    """Write the manifest, sorted for a stable, diff-friendly file."""
    MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    document = {
        "description": (
            "Provenance record of source datasets retrieved by the COMP5339 "
            "Assignment 1 acquisition notebook."
        ),
        "generated_at": utc_now_iso(),
        "artefacts": dict(sorted(manifest.items())),
    }
    MANIFEST_PATH.write_text(json.dumps(document, indent=2) + "\n", encoding="utf-8")
    log(f"Manifest written to {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
    return MANIFEST_PATH


# One session and one manifest are shared by both sources below.
session = build_session()
manifest = load_manifest()
log(f"Session ready; manifest holds {len(manifest)} existing entrie(s).")

[18:13:37] Session ready; manifest holds 2 existing entrie(s).


<a id="source1"></a>
## 3. Source 1 — Transport for NSW EV charging locations

The brief allows this file to be obtained either from the Transport for NSW open data portal
or via data.gov.au, noting that the TfNSW portal requires a free login. This notebook uses the
data.gov.au route, for a reason worth stating precisely.

data.gov.au does not host its own copy of the file. Its CKAN record is a **catalogue entry
that points at the TfNSW-hosted resource**, so the URL discovered below resolves to
`opendata.transport.nsw.gov.au` — the file is retrieved from Transport for NSW itself. What
data.gov.au provides is a *public API for discovering that URL*, avoiding the authenticated
browser session the portal's own interface expects.

That distinction matters for reproducibility. An authenticated download cannot be reproduced
by a reader who does not hold the same credentials, and embedding credentials in a submitted
notebook is not an acceptable alternative. Discovering the URL through the public catalogue
keeps the retrieval fully automated and runnable by anyone, which is what the brief requires.

### 3.1 What the CKAN API actually returns

Choosing the right resource needs care, for two reasons that are visible in the table below:

* the dataset also contains a PDF data dictionary, a **superseded 2021 CSV**, and an unrelated
  NRMA usage spreadsheet — none of which should be downloaded; and
* CKAN's `last_modified` field is **null for every resource**, so the obvious approach of
  "sort by date, take the newest" does not work.

The release date is therefore parsed out of the filename convention TfNSW uses,
`ev_YYYYMMDD.csv`.

In [6]:
def parse_release_date(url: str):
    """
    Extract the release date from a TfNSW resource URL, or None if it has none.

    TfNSW encodes the release in the filename (`.../ev_20251216.csv`). This is the
    only usable version marker, since CKAN's own `last_modified` is null for every
    resource in this dataset.
    """
    match = TFNSW_CSV_DATE_PATTERN.search(url)
    if match is None:
        return None
    try:
        return datetime.strptime(match.group("date"), "%Y%m%d").date()
    except ValueError:
        # A filename that looks like a date but isn't (e.g. ev_20251345.csv).
        return None


response = session.get(
    f"{CKAN_API_BASE}/package_show",
    params={"id": CKAN_PACKAGE_ID},
    timeout=REQUEST_TIMEOUT,
)
response.raise_for_status()
ckan_payload = response.json()

resources = pd.DataFrame(ckan_payload["result"]["resources"])
resources["filename"] = resources["url"].str.rsplit("/", n=1).str[-1]
resources["parsed_release_date"] = resources["url"].map(parse_release_date)

log(f"CKAN returned {len(resources)} resources for '{CKAN_PACKAGE_ID}'")
resources[["name", "format", "last_modified", "filename", "parsed_release_date"]]

[18:13:37] CKAN returned 4 resources for 'nsw-2-ev-charging-locations'


,name,format,last_modified,filename,parsed_release_date
0,EV Charging Locations in NSW,CSV,None,ev_20251216.csv,2025-12-16
1,EV Charging Locations documentation,PDF,None,ev-charging-locations-v2.1.pdf,None
2,EV Charging Stations in NSW - Not updated,CSV,None,electric-vehicle-charging-stations-nsw-20211207.csv,None
3,NRMA Co-Funded Charger Usage Report - Not updated,XLS,None,nrma-co-funded-charger-usage-report-october-2022.xlsx,None


Only two resources yield a parsable release date, and just one of those falls in the target
month. The selection rule below prefers the newest CSV released in **December 2025**, as the
brief requires, and falls back to the newest release of any month with a warning if TfNSW ever
withdraws it.

In [7]:
def discover_tfnsw_csv(payload: dict) -> dict:
    """
    Return the CKAN resource dictionary that should be downloaded.

    Selection rule, in order of preference:
      1. the newest dated CSV released in the target month (December 2025);
      2. otherwise the newest dated CSV of any month, with a warning.
    """
    if not payload.get("success"):
        raise RuntimeError(f"CKAN reported failure for package '{CKAN_PACKAGE_ID}'")

    # Keep only CSV resources whose filename carries a parsable release date;
    # this filters out the PDF data dictionary and the NRMA spreadsheet.
    dated = [
        (parse_release_date(r.get("url", "")), r)
        for r in payload["result"].get("resources", [])
        if parse_release_date(r.get("url", "")) is not None
    ]
    if not dated:
        raise RuntimeError(
            "No CSV matching the 'ev_YYYYMMDD.csv' convention was found - "
            "the dataset's publication format may have changed."
        )

    dated.sort(key=lambda pair: pair[0], reverse=True)
    in_target_month = [
        (d, r) for d, r in dated
        if d.year == TARGET_RELEASE_YEAR and d.month == TARGET_RELEASE_MONTH
    ]

    if in_target_month:
        release_date, resource = in_target_month[0]
        log(f"Selected the {release_date} release, as required by the brief.")
    else:
        release_date, resource = dated[0]
        log(
            f"WARNING: no release for {TARGET_RELEASE_YEAR}-{TARGET_RELEASE_MONTH:02d}; "
            f"falling back to the most recent available ({release_date})."
        )

    resource["_release_date"] = release_date.isoformat()
    return resource


# Discovery is the fragile part; the download itself is not. Falling back to the
# pinned URL keeps the notebook runnable if data.gov.au is down, and the manifest
# records that this is what happened.
try:
    selected = discover_tfnsw_csv(ckan_payload)
    tfnsw_url = selected["url"]
    tfnsw_release_date = selected["_release_date"]
    tfnsw_discovery = "data.gov.au CKAN API (action/package_show)"
except (requests.RequestException, ValueError, RuntimeError, KeyError) as error:
    log(f"ERROR: CKAN discovery failed ({error}); using the pinned fallback URL.")
    tfnsw_url = TFNSW_FALLBACK_URL
    tfnsw_release_date = "2025-12-16"
    tfnsw_discovery = "pinned fallback URL (CKAN discovery unavailable)"

print(f"\nURL:      {tfnsw_url}")
print(f"Released: {tfnsw_release_date}")

[18:13:37] Selected the 2025-12-16 release, as required by the brief.

URL:      https://opendata.transport.nsw.gov.au/data/dataset/be1c4de4-4517-4bd0-8a09-2965ddfc7179/resource/7bbb6461-e52d-4fe7-ace4-a15c30198de0/download/ev_20251216.csv
Released: 2025-12-16


### 3.2 Download and validate

The downloaded file is checked against the twelve expected columns before anything else uses
it. That validation matters more than usual here, because **the server misreports the file
type**: TfNSW returns the Content-Type of an Excel workbook
(`application/vnd.openxmlformats-officedocument.spreadsheetml.sheet`) for a file whose body is
plain UTF-8 CSV. Trusting the header would hand the file to the wrong parser, so the content
itself is inspected instead.

In [8]:
def validate_ev_csv(path: Path) -> int:
    """
    Confirm the file really is the expected CSV, and return its row count.

    Read with `utf-8-sig`, not `utf-8`: the file begins with a UTF-8 byte-order
    mark, and reading it as plain utf-8 leaves a stray BOM character glued to the
    first column name - which silently breaks every later reference to it.
    """
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(handle)
        try:
            header = next(reader)
        except StopIteration as error:
            raise ValueError(f"{path.name} is empty") from error
        row_count = sum(1 for _ in reader)

    missing = [c for c in EXPECTED_TFNSW_COLUMNS if c not in header]
    if missing:
        raise ValueError(f"{path.name} is missing column(s): {', '.join(missing)}")

    unexpected = [c for c in header if c not in EXPECTED_TFNSW_COLUMNS]
    if unexpected:
        # Not fatal, but reported so the change is handled deliberately in Task 2.
        log(f"WARNING: unexpected column(s) in {path.name}: {', '.join(unexpected)}")
    if row_count == 0:
        raise ValueError(f"{path.name} has a header but no data rows")

    log(f"Validated {path.name}: {len(header)} columns, {row_count:,} data rows.")
    return row_count


EV_CSV_PATH = RAW_TFNSW_DIR / Path(tfnsw_url).name

if is_unchanged(manifest, "tfnsw_ev_charging_locations", EV_CSV_PATH):
    log(f"{EV_CSV_PATH.name} already present and unchanged; skipping download.")
else:
    download_file(session, tfnsw_url, EV_CSV_PATH)

ev_row_count = validate_ev_csv(EV_CSV_PATH)

record_artefact(
    manifest,
    "tfnsw_ev_charging_locations",
    path=EV_CSV_PATH,
    url=tfnsw_url,
    source=(
        "Transport for NSW - EV Charging Locations in NSW "
        "(mirrored on data.gov.au, dataset id 'nsw-2-ev-charging-locations')"
    ),
    discovery_method=tfnsw_discovery,
    extra={
        "release_date": tfnsw_release_date,
        "format": "CSV (UTF-8 with BOM)",
        "data_rows": ev_row_count,
        "note": (
            "Server reports an Excel Content-Type but serves CSV; read with "
            "encoding='utf-8-sig'."
        ),
    },
)
print(f"\n-> {EV_CSV_PATH.relative_to(PROJECT_ROOT)}")

[18:13:37] ev_20251216.csv already present and unchanged; skipping download.
[18:13:37] Validated ev_20251216.csv: 12 columns, 1,958 data rows.



-> data/raw/tfnsw/ev_20251216.csv


<a id="source2"></a>
## 4. Source 2 — ABS ASGS Edition 4 SA4 digital boundaries

The ABS publishes its boundary files as ZIP archives linked from an ordinary HTML page — there
is no API and no CKAN mirror. The page is therefore fetched and parsed with BeautifulSoup.

Links are matched against the ABS **filename convention** `SA4_<year>_AUST_SHP_<datum>.zip`
rather than one literal filename, so a change of reference year or datum is picked up
automatically and the year actually retrieved is written to the manifest.

In [9]:
def discover_sa4_zip_url(session: requests.Session) -> str:
    """
    Scrape the ABS boundary-files page and return the SA4 archive URL.

    Every download on that page is a plain `<a href="...zip">`, so no browser or
    JavaScript execution is needed. If the ABS ever listed more than one SA4
    archive, the most recent reference year wins.
    """
    page = session.get(ABS_BOUNDARY_PAGE_URL, timeout=REQUEST_TIMEOUT)
    page.raise_for_status()
    soup = BeautifulSoup(page.text, "lxml")

    matches = []
    for anchor in soup.find_all("a", href=True):
        match = ABS_SA4_ZIP_PATTERN.search(anchor["href"])
        if match:
            # Links are site-relative, so resolve them against the page URL
            # rather than concatenating strings.
            matches.append((int(match.group("year")),
                            urljoin(ABS_BOUNDARY_PAGE_URL, anchor["href"])))

    if not matches:
        raise RuntimeError(
            "No SA4 shapefile link found on the ABS page - its structure or "
            "filename convention may have changed."
        )

    matches.sort(key=lambda pair: pair[0], reverse=True)
    year, url = matches[0]
    log(f"Found the SA4 {year} boundary archive.")
    return url


sa4_url = discover_sa4_zip_url(session)
print(sa4_url)

[18:13:37] Found the SA4 2026 boundary archive.
https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs/edition-4-july-2026-june-2031/access-and-downloads/digital-boundary-files/SA4_2026_AUST_SHP_GDA2020.zip


The archive is unpacked with **each member's path checked first**. A ZIP entry is free to
contain `..` or an absolute path, which `extractall` would follow and write *outside* the
target folder (the "zip slip" problem). Rejecting those names keeps extraction contained.

Afterwards the four mandatory shapefile components — `.shp`, `.shx`, `.dbf`, `.prj` — are
confirmed present, since a shapefile is a *set* of sibling files rather than one file.

In [10]:
def extract_shapefile(archive_path: Path, target_dir: Path) -> Path:
    """Extract the archive safely and return the path of the .shp inside it."""
    if not zipfile.is_zipfile(archive_path):
        raise ValueError(
            f"{archive_path.name} is not a valid ZIP - the download may have "
            "returned an error page instead of the file."
        )
    target_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(archive_path) as archive:
        members = archive.namelist()
        for member in members:
            member_path = Path(member)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Refusing to extract unsafe path: {member!r}")
        archive.extractall(target_dir)

    log(f"Extracted {len(members)} file(s) to {target_dir.relative_to(PROJECT_ROOT)}")

    shp_files = sorted(target_dir.rglob("*.shp"))
    if not shp_files:
        raise ValueError(f"No .shp file found inside {archive_path.name}")

    shapefile = shp_files[0]
    present = {p.suffix.lower() for p in shapefile.parent.glob(f"{shapefile.stem}.*")}
    missing = REQUIRED_SHAPEFILE_SUFFIXES - present
    if missing:
        raise ValueError(f"Incomplete shapefile; missing {', '.join(sorted(missing))}")

    log(f"Shapefile components present: {', '.join(sorted(present))}")
    return shapefile


def read_projection(shapefile: Path) -> str:
    """
    Return the CRS name from the shapefile's `.prj` sidecar file.

    Worth recording at acquisition time, because the two datasets do NOT share a
    coordinate reference system: the ABS boundaries are GDA2020 while the TfNSW
    latitude/longitude columns are WGS84. Task 2's spatial join must reconcile
    them, and this is the evidence of what was actually supplied.
    """
    prj_text = shapefile.with_suffix(".prj").read_text(encoding="utf-8", errors="replace")
    # A .prj holds one WKT string, e.g. GEOGCS["GDA2020", ...
    first_quote = prj_text.find('"')
    second_quote = prj_text.find('"', first_quote + 1)
    return prj_text[first_quote + 1:second_quote] if first_quote != -1 else "unknown"


SA4_ZIP_PATH = RAW_ABS_DIR / Path(sa4_url).name

if is_unchanged(manifest, "abs_asgs_sa4_boundaries", SA4_ZIP_PATH):
    log(f"{SA4_ZIP_PATH.name} already present and unchanged; skipping download.")
else:
    download_file(session, sa4_url, SA4_ZIP_PATH)

# Re-extract whenever the extracted folder is absent; this also repairs a run
# interrupted between downloading and unpacking.
already_extracted = sorted(SA4_EXTRACT_DIR.rglob("*.shp"))
SA4_SHAPEFILE_PATH = (
    already_extracted[0] if already_extracted
    else extract_shapefile(SA4_ZIP_PATH, SA4_EXTRACT_DIR)
)

sa4_crs = read_projection(SA4_SHAPEFILE_PATH)
log(f"Shapefile CRS: {sa4_crs}")

record_artefact(
    manifest,
    "abs_asgs_sa4_boundaries",
    path=SA4_ZIP_PATH,
    url=sa4_url,
    source=(
        "Australian Bureau of Statistics - Australian Statistical Geography "
        "Standard (ASGS) Edition 4, July 2026 - June 2031, SA4 digital boundaries"
    ),
    discovery_method=(
        "HTML scrape of the ABS digital boundary files page (BeautifulSoup link matching)"
    ),
    extra={
        "format": "ESRI Shapefile (ZIP archive)",
        "extracted_to": str(SA4_EXTRACT_DIR.relative_to(PROJECT_ROOT)),
        "shapefile": str(SA4_SHAPEFILE_PATH.relative_to(PROJECT_ROOT)),
        "crs": sa4_crs,
    },
)
print(f"\n-> {SA4_SHAPEFILE_PATH.relative_to(PROJECT_ROOT)}")

[18:13:37] SA4_2026_AUST_SHP_GDA2020.zip already present and unchanged; skipping download.
[18:13:37] Shapefile CRS: GDA2020

-> data/raw/abs/sa4_shapefile/SA4_2026_AUST_GDA2020.shp


<a id="manifest"></a>
## 5. Provenance manifest

Both files are now recorded. Re-running this notebook re-hashes what is on disk and skips the
downloads only if the checksums still match, so a corrupted local copy is repaired rather than
trusted.

In [11]:
save_manifest(manifest)

pd.DataFrame(manifest).T[
    ["source", "discovery_method", "bytes", "sha256", "retrieved_at"]
]

[18:13:37] Manifest written to data/raw/manifest.json


,source,discovery_method,bytes,sha256,retrieved_at
abs_asgs_sa4_boundaries,Australian Bureau of Statistics - Australian Statistical...,HTML scrape of the ABS digital boundary files page (Beau...,29543412,d2df15ee57dac089457fd6ff27e342d4f6286ab0dbc587fff20cf056...,2026-09-22T08:13:37+00:00
tfnsw_ev_charging_locations,Transport for NSW - EV Charging Locations in NSW (mirror...,data.gov.au CKAN API (action/package_show),283033,43970e7751b951ab459a1be7bfd6141756c60ff8aa797debf11c5a59...,2026-09-22T08:13:37+00:00


<a id="profile"></a>
## 6. Data quality profile

The remainder of the notebook profiles what was retrieved. These figures identify the data
quality issues, and they determine the cleaning strategy that Task 2 implements.

In [12]:
ev = pd.read_csv(EV_CSV_PATH, encoding="utf-8-sig")   # utf-8-sig strips the BOM
print(f"{ev.shape[0]:,} rows x {ev.shape[1]} columns")
ev.head()

1,958 rows x 12 columns


,OBJECTID,Station_name,Station_address,Operator,Number_of_plugs,Charger_Type,Charger_rating,Latitude,Longitude,LGANAME,PCODE,Source
0,NaN,NaN,", Muswellbrook, 2333",EVUp,2,AC,22 kW,-32.262242,150.890139,Muswellbrook Shire Council,2333,Existing Destination Chargers
1,NaN,NaN,"01 Wallgrove Road, Sydney, 2766",BP,4,DC,150 kW,-33.811004,150.849597,Blacktown City Council,2766,Existing Fast Chargers
2,NaN,NaN,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,4,DC,50 kW,-30.511874,151.669395,Central Darling Shire Council,2350,TfNSW Regional
3,NaN,NaN,"1 Balfour St, Sydney, 2070",Chargefox,7,AC,22 kW,-33.774101,151.167035,Ku-ring-gai Council,2070,Existing Destination Chargers
4,NaN,NaN,"1 Bay Ln, Byron Bay, 2481",Tesla,2,AC,19 kW,-28.641819,153.613633,Byron Shire Council,2481,Existing Destination Chargers


### 6.1 Missing values

`Station_name` is absent from most records. `LGANAME`, `PCODE` and `Source` are null in
**the same 121 rows**, which suggests those records entered the dataset through a different
process from the rest.

In [13]:
profile = pd.DataFrame({
    "dtype": ev.dtypes.astype(str),
    "nulls": ev.isna().sum(),
    "null_pct": (ev.isna().mean() * 100).round(1),
    "distinct": ev.nunique(),
})
profile

,dtype,nulls,null_pct,distinct
OBJECTID,float64,1837,93.8,121
Station_name,object,1438,73.4,477
Station_address,object,0,0.0,1924
Operator,object,0,0.0,50
Number_of_plugs,int64,0,0.0,18
Charger_Type,object,0,0.0,3
Charger_rating,object,0,0.0,46
Latitude,float64,0,0.0,1937
Longitude,float64,0,0.0,1935
LGANAME,object,121,6.2,129


### 6.2 The finding that shapes Task 3

`Charger_Type` carries a third value, `Upcoming`, which describes a charger's *status* rather
than its type — Task 2 has to decide explicitly whether those records are in scope.

More consequentially: of the **433 DC fast chargers** that Task 3 must augment, **only one has
a station name**. Name-based matching against an external source (Open Charge Map, operator
websites) is therefore impossible, which makes **coordinates the only reliable join key**.

In [14]:
print(ev["Charger_Type"].value_counts().to_string(), "\n")

name_coverage = ev.groupby("Charger_Type")["Station_name"].agg(
    records="size", with_name="count"
)
name_coverage["with_name_pct"] = (
    name_coverage["with_name"] / name_coverage["records"] * 100
).round(1)
name_coverage

Charger_Type
AC          1427
DC           433
Upcoming      98 



,records,with_name,with_name_pct
Charger_Type,,,
AC,1427,515,36.1
DC,433,1,0.2
Upcoming,98,4,4.1


### 6.3 Inconsistent operator naming

50 distinct operator strings stand for roughly 35 real operators. Three separate problems are
tangled together in this one column:

1. **Case and punctuation variants** that a normalisation function can merge on its own —
   `ChargeHub`/`Charge Hub`, `Non-networked`/`Non-Networked`.
2. **Trailing whitespace** — `'BP Australia '` and `'Tesla Motors '` both carry a trailing
   space, so they are distinct strings from their trimmed forms even before any other
   difference is considered.
3. **Truncation at 13–14 characters**, which the output below effectively proves: the dataset
   contains *both* `Viva Energy A` and `Viva Energy Australia`, and both `PLUS ES` and
   `PLUS ES Manag`. A fixed-width field somewhere upstream is cutting names short. These
   cannot be repaired by normalisation and need a manual lookup table.

A caution on the heuristic below: grouping by shared first word is a *lead generator*, not an
answer. It correctly pairs `Evie`/`Evie Networks`, but it also pairs `Charge Hub` with
`Charge OS`, which are unrelated companies. Every proposed merge needs human review before
Task 2 applies it — which is exactly why the mapping belongs in an explicit lookup table
rather than in an automatic rule.

In [15]:
print(f"{ev['Operator'].nunique()} distinct operator values\n")

# Group operators by a crude normalised key to surface likely duplicate entities.
normalised_key = ev["Operator"].str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
variant_groups = (
    pd.DataFrame({"operator": ev["Operator"], "key": normalised_key})
      .drop_duplicates()
      .groupby("key")["operator"]
      .apply(list)
)
print("Variants differing only by case or punctuation:")
for names in variant_groups[variant_groups.map(len) > 1]:
    print("   ", names)

# Operators sharing a leading word are likely the same entity written two ways.
print("\nOperators sharing a first word (possible same entity):")
first_word = pd.Series(ev["Operator"].unique()).str.split().str[0].str.lower()
for word, group in pd.Series(ev["Operator"].unique()).groupby(first_word):
    if len(group) > 1:
        print("   ", sorted(group))

# Trailing/leading whitespace makes otherwise identical names distinct strings.
whitespace_affected = [n for n in ev["Operator"].unique() if n != n.strip()]
print("\nOperator values with leading/trailing whitespace:", whitespace_affected)

# Where one operator name is a strict prefix of another, the pair is either a
# naming variant ('BP' / 'BP Australia') or genuine truncation
# ('Viva Energy A' / 'Viva Energy Australia'). The test cannot tell them apart,
# so it reports the candidates and flags which ones end mid-word - a name cut
# off at a partial word is the signature of truncation rather than abbreviation.
unique_operators = sorted(ev["Operator"].unique(), key=len)
prefix_pairs = [
    (short, longer)
    for short in unique_operators
    for longer in unique_operators
    if short != longer and longer.startswith(short.rstrip()) and len(short.strip()) <= 20
]
print("\nPrefix pairs (naming variant or truncation - needs review):")
for short, longer in prefix_pairs:
    # If the longer name continues the short one mid-word, the short form was cut.
    cut_mid_word = len(longer) > len(short.rstrip()) and longer[len(short.rstrip())] != " "
    marker = "  <- truncated mid-word" if cut_mid_word else ""
    print(f"    {short!r:<24} -> {longer!r}{marker}")

print()
ev["Operator"].value_counts().head(15)

50 distinct operator values

Variants differing only by case or punctuation:
    ['ChargeHub', 'Charge Hub']
    ['Non-networked', 'Non-Networked']

Operators sharing a first word (possible same entity):
    ['BP', 'BP Australia ']
    ['Charge Hub', 'Charge OS']
    ['Evie', 'Evie Networks']
    ['Non-Networked', 'Non-networked']
    ['NRMA', 'NRMA Electric']
    ['PLUS ES', 'PLUS ES Manag']
    ['Porsche Destination Charging', 'Porsche Smart Mobility']
    ['Tesla', 'Tesla Motors ']
    ['Viva Energy A', 'Viva Energy Australia']

Operator values with leading/trailing whitespace: ['BP Australia ', 'Tesla Motors ']

Prefix pairs (naming variant or truncation - needs review):
    'BP'                     -> 'BP Australia '
    'NRMA'                   -> 'NRMA Electric'
    'Evie'                   -> 'Evie Networks'
    'Tesla'                  -> 'Tesla Motors '
    'PLUS ES'                -> 'PLUS ES Manag'
    'Viva Energy A'          -> 'Viva Energy Australia'  <- truncated mid-wo

Operator
Exploren         301
Tesla            260
Chargefox        254
Non-networked    217
PLUS ES          150
EVX              111
Evie              84
NRMA              78
JOLT              49
EVUp              38
Everty            36
Ampol             32
BP                32
BP Australia      28
Tesla Motors      27
Name: count, dtype: int64

### 6.4 Inconsistent charger attributes

`Charger_rating` mixes at least four formats in a single column: a value with units (`22 kW`),
a bare number (`22`), a non-numeric placeholder (`AC`, in 522 rows), and compound
multi-connector descriptions (`2x350kW & 2x175kW`).

Turning this into something queryable is a genuine **schema design decision** for Task 4 — not
just a string cleanup — because a compound value describes several connectors at one site.

In [16]:
print(f"{ev['Charger_rating'].nunique()} distinct rating strings\n")

def rating_format(value) -> str:
    """Classify a raw Charger_rating string by the format it uses."""
    value = str(value).strip()
    if re.fullmatch(r"\d+(\.\d+)?\s*kW", value, re.IGNORECASE):
        return "number + unit  (e.g. '22 kW')"
    if re.fullmatch(r"\d+(\.\d+)?", value):
        return "bare number    (e.g. '22')"
    if re.search(r"\dx\s*\d", value, re.IGNORECASE):
        return "compound       (e.g. '2x350kW & 2x175kW')"
    if re.fullmatch(r"[A-Za-z ]+", value):
        return "non-numeric    (e.g. 'AC')"
    return "other"

print(ev["Charger_rating"].map(rating_format).value_counts().to_string())
print()
ev["Charger_rating"].value_counts().head(12)

46 distinct rating strings

Charger_rating
number + unit  (e.g. '22 kW')                1315
non-numeric    (e.g. 'AC')                    522
compound       (e.g. '2x350kW & 2x175kW')      99
bare number    (e.g. '22')                     22



Charger_rating
22 kW                634
AC                   522
6 kW                 112
50 kW                 86
2x350kW & 2x175kW     85
75 kW                 80
7 kW                  66
25 kW                 52
175 kW                46
150 kW                44
11 kW                 31
125 kW                19
Name: count, dtype: int64

### 6.5 Duplicates and coordinate sanity

There are no exactly duplicated rows, but some coordinate pairs recur. These must be
*examined* rather than blindly dropped: one physical site can legitimately host separate AC and
DC units, which are genuinely different records.

Every coordinate falls inside the NSW bounding box, so the geometry itself is sound and the
spatial join in Task 2 should not encounter stray points.

In [17]:
print("Exact duplicate rows:                 ", ev.duplicated().sum())
print("Duplicate (lat, lon):                 ",
      ev.duplicated(subset=["Latitude", "Longitude"]).sum())
print("Duplicate (lat, lon, operator, type): ",
      ev.duplicated(subset=["Latitude", "Longitude", "Operator", "Charger_Type"]).sum())

NSW_LAT = (-37.6, -28.1)      # approximate NSW bounding box
NSW_LON = (140.9, 153.7)
outside = ev[~ev["Latitude"].between(*NSW_LAT) | ~ev["Longitude"].between(*NSW_LON)]

print(f"\nLatitude range:  {ev.Latitude.min():.4f} to {ev.Latitude.max():.4f}")
print(f"Longitude range: {ev.Longitude.min():.4f} to {ev.Longitude.max():.4f}")
print(f"Points outside the NSW bounding box: {len(outside)}")

# The duplicated coordinates, shown so Task 2 can decide case by case.
ev[ev.duplicated(subset=["Latitude", "Longitude"], keep=False)].sort_values(
    ["Latitude", "Longitude"]
)[["Station_address", "Operator", "Charger_Type", "Charger_rating", "Number_of_plugs"]].head(10)

Exact duplicate rows:                  0
Duplicate (lat, lon):                  20
Duplicate (lat, lon, operator, type):  11

Latitude range:  -37.1115 to -28.1686
Longitude range: 141.4601 to 153.6159
Points outside the NSW bounding box: 0


,Station_address,Operator,Charger_Type,Charger_rating,Number_of_plugs
604,520 David St\nAlbury NSW 2640\nAustralia,Exploren,AC,AC,4
1087,"520 David St, Albury NSW 2640, Australia",Exploren,AC,7,2
530,406 Moppett St\nHay NSW 2711\nAustralia,Exploren,AC,AC,4
1073,"406 Moppett St, Hay NSW 2711, Australia",Exploren,AC,22,2
245,"19 Princes Hwy, Figtree NSW 2525, Australia",Tesla,DC,175 kW,6
1088,"19 Princes Hwy, Figtree NSW 2525, Australia",Tesla Motors,DC,2x350kW & 2x175kW,6
1150,"19 Princes Hwy, Figtree NSW 2525",Non-networked,AC,AC,2
967,"University of Wollongong, Northfields Avenue, Wollongong...",Chargefox,DC,175 kW,6
1061,"University of Wollongong, Northfields Avenue, Wollongong...",University of,DC,2x350kW & 2x175kW,6
1775,Early Start Discovery Space (Building 21) University of ...,Chargefox,AC,AC,4


### 6.6 Address formats

`Station_address` appears in at least three formats, and **733 of the 1,958 records contain
embedded newlines** — enough that any naive string comparison against an external source will
fail on more than a third of the data.

The locality component is separately unreliable: **399 records** give the locality as the
placeholder `Sydney` rather than the true suburb. `293 Belmore Rd, Sydney, 2210` is in
Riverwood, some 15 km from the CBD.

Address matching in Task 3 therefore cannot depend on the locality field; the postcode and the
street line are the trustworthy parts.

In [18]:
for address in ev["Station_address"].sample(6, random_state=1):
    print(repr(address))

print("\nRecords whose locality is the placeholder 'Sydney':",
      ev["Station_address"].str.contains(r",\s*Sydney\s*,", regex=True, na=False).sum())
print("Records containing an embedded newline:            ",
      ev["Station_address"].str.contains("\n", na=False).sum())

'McFarlane St, Sydney, 2160'
'697 Wollombi Rd\nBroke NSW 2330\nAustralia'
'154 Beach Rd, Batemans Bay NSW 2536'
'53 Macquarie Rd\nCardiff NSW 2285\nAustralia'
'76 Wingewarra St, Dubbo , 2830'
'Lauder St, Tumbarumba, 2653'

Records whose locality is the placeholder 'Sydney': 399
Records containing an embedded newline:             733


<a id="duckdb"></a>
## 7. SA4 boundaries in DuckDB

The shapefile is read with DuckDB's `spatial` extension rather than GeoPandas. Two reasons:

* it keeps the dependency list small (GeoPandas pulls in GDAL, Fiona and Shapely); and
* more importantly, the point-in-polygon join in Task 2 and the final storage in Task 4 then
  happen in the **same engine the assignment requires**, instead of moving data between two.

In [19]:
import duckdb

con = duckdb.connect()                       # in-memory; Task 4 will use a file
con.execute("INSTALL spatial; LOAD spatial;")

con.execute(
    f"CREATE OR REPLACE VIEW sa4 AS SELECT * FROM ST_Read('{SA4_SHAPEFILE_PATH.as_posix()}')"
)
con.execute("DESCRIBE sa4").df()

,column_name,column_type,null,key,default,extra
0,OGC_FID,BIGINT,YES,None,None,None
1,SA4_CODE26,VARCHAR,YES,None,None,None
2,SA4_NAME26,VARCHAR,YES,None,None,None
3,CHG_FLAG26,VARCHAR,YES,None,None,None
4,CHG_LBL26,VARCHAR,YES,None,None,None
5,GCC_CODE26,VARCHAR,YES,None,None,None
6,GCC_NAME26,VARCHAR,YES,None,None,None
7,STE_CODE26,VARCHAR,YES,None,None,None
8,STE_NAME26,VARCHAR,YES,None,None,None
9,AUS_CODE26,VARCHAR,YES,None,None,None


The geometry column reports **EPSG:7844** (GDA2020), whereas the TfNSW latitude/longitude
columns are WGS84. The two datasets do not share a coordinate reference system — something
Task 2's spatial join has to reconcile explicitly rather than assume away.

The ABS publishes SA4 boundaries for the whole of Australia, so NSW is filtered in Task 2:
**30 of the 108 SA4 regions**.

In [20]:
con.execute('''
    SELECT STE_NAME26 AS state, COUNT(*) AS sa4_regions
    FROM sa4
    GROUP BY state
    ORDER BY sa4_regions DESC
''').df()

,state,sa4_regions
0,New South Wales,30
1,Queensland,21
2,Victoria,19
3,Western Australia,12
4,South Australia,9
5,Tasmania,6
6,Northern Territory,4
7,Australian Capital Territory,3
8,Other Territories,3
9,Outside Australia,1


In [21]:
con.execute('''
    SELECT SA4_CODE26, SA4_NAME26, GCC_NAME26, ROUND(AREASQKM26, 1) AS area_sqkm
    FROM sa4
    WHERE STE_NAME26 = 'New South Wales'
    ORDER BY SA4_CODE26
''').df()

,SA4_CODE26,SA4_NAME26,GCC_NAME26,area_sqkm
0,101,Capital Region,Rest of NSW,51896.2
1,102,Central Coast,Greater Sydney,1681.0
2,103,Central West,Rest of NSW,70297.1
3,104,Coffs Harbour - Grafton,Rest of NSW,13229.8
4,105,Far West and Orana,Rest of NSW,339355.6
5,106,Hunter Valley exc Newcastle,Rest of NSW,21491.3
6,107,Illawarra,Rest of NSW,1539.2
7,108,Mid North Coast,Rest of NSW,18851.5
8,109,Murray,Rest of NSW,97796.5
9,110,New England and North West,Rest of NSW,99139.9


In [22]:
con.close()
session.close()
log("Task 1 complete.")

[18:13:37] Task 1 complete.


<a id="summary"></a>
## 8. Summary and outputs

### What was retrieved

| Dataset | File | Size | Records |
|---|---|---|---|
| TfNSW EV charging locations (16 Dec 2025) | `data/raw/tfnsw/ev_20251216.csv` | 283 KB | 1,958 chargers |
| ABS ASGS Ed. 4 SA4 boundaries (2026, GDA2020) | `data/raw/abs/SA4_2026_AUST_SHP_GDA2020.zip` | 29.5 MB | 108 SA4 regions (30 in NSW) |

Both URLs were **discovered at run time** — via the CKAN API and by parsing the ABS page —
rather than hard-coded, so the notebook keeps working when the publishers issue a new release.
Nothing was downloaded by hand.

### Reproducibility features

* **Retries with exponential backoff** on connection errors and 429/5xx responses; a 404 is a
  real answer and is raised rather than retried.
* **Atomic downloads** — each file streams to a temporary `.part` and is renamed into place
  only on success, so an interrupted run cannot leave a truncated file behind.
* **Checksum-verified caching** — a re-run re-hashes what is on disk and skips the download
  only if it matches the manifest.
* **Content validation** — the CSV header is checked against the twelve expected columns and
  its row count confirmed non-zero; the ZIP is verified as a real archive containing a complete
  shapefile.
* **Safe extraction** — archive members with `..` or absolute paths are rejected.

### Data quality issues identified

| # | Issue | Detail |
|---|---|---|
| 1 | Missing station names | Null in 1,438 records — **including 432 of 433 DC chargers** |
| 2 | Inconsistent operator naming | 50 values for ~35 entities; case/punctuation variants, trailing whitespace, and names truncated at 13–14 chars |
| 3 | Inconsistent charger attributes | `Charger_rating` mixes 4 formats across 46 distinct strings (1,315 unit-suffixed, 522 non-numeric, 99 compound, 22 bare) |
| 4 | Status mixed into type | `Charger_Type` includes `Upcoming` (98 rows) |
| 5 | Correlated missingness | `LGANAME`, `PCODE`, `Source` null in the same 121 rows |
| 6 | Duplicate coordinates | 20 records share a location with another record |
| 7 | Address inconsistency | 3 formats; 733 records contain embedded newlines; 399 use `Sydney` as a placeholder locality |
| 8 | CRS mismatch | ABS boundaries are GDA2020 (EPSG:7844); TfNSW columns are WGS84 |

### Outputs of this stage

Running this notebook produces the following, all under `data/raw/`. These are the inputs to
Task 2, which reads them directly rather than repeating the retrieval.

| File | Contents |
|---|---|
| `tfnsw/ev_20251216.csv` | 1,958 raw EV charger records, unmodified as downloaded |
| `abs/SA4_2026_AUST_SHP_GDA2020.zip` | ABS SA4 boundary archive, unmodified as downloaded |
| `abs/sa4_shapefile/` | The extracted shapefile: 108 SA4 boundary polygons |
| `manifest.json` | Provenance record — source URL, discovery method, retrieval timestamp, byte size and SHA-256 checksum for each file |

Within this notebook these are held in `EV_CSV_PATH`, `SA4_SHAPEFILE_PATH` and
`MANIFEST_PATH`.

Source files are written to `data/raw/` and never modified in place; cleaned and derived data
are written to `data/interim/` and `data/processed/` by later stages. Keeping the raw layer
immutable means any result can be reproduced from the downloaded files alone, and that a bug
in a later stage cannot corrupt the evidence it was derived from.

### Implications for the remaining tasks

Three findings above constrain how the later stages must be approached:

* **Augmentation cannot use station names.** With 432 of 433 DC chargers unnamed, matching to
  an external source such as Open Charge Map has to be driven by coordinates, with address
  fields serving only as a secondary check.
* **Address text is not a reliable matching key either.** 733 records contain embedded
  newlines and 399 carry a placeholder locality, so only the street line and postcode can be
  trusted.
* **The spatial join must handle two coordinate reference systems.** The ABS boundaries are
  published in GDA2020 (EPSG:7844) while the TfNSW coordinates are WGS84, so the join has to
  reconcile them explicitly rather than assume they align.